# Introduction

It looks like we're back to our usual classification problem. The problem in hands is that we have to predict whether a horse will live, die, or get another fate based on their health history. The metric we have is micro-averaged F1-score, which means that some classes will get bias based on their frequency.

If you want to read the description of the original dataset, you can visit this page: https://www.kaggle.com/datasets/yasserh/horse-survival-dataset.

# Loading Libraries and Datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from category_encoders import OneHotEncoder, GLMMEncoder, TargetEncoder, CatBoostEncoder
from sklearn import set_config
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.metrics import roc_auc_score, roc_curve, make_scorer, f1_score
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, IterativeImputer, KNNImputer
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.preprocessing import FunctionTransformer, StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.gaussian_process import GaussianProcessClassifier
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

sns.set_theme(style = 'white', palette = 'viridis')
pal = sns.color_palette('viridis')

pd.set_option('display.max_rows', 100)
set_config(transform_output = 'pandas')
pd.options.mode.chained_assignment = None

In [ ]:
train = pd.read_csv(r'/kaggle/input/playground-series-s3e22/train.csv', index_col = 'id')
test = pd.read_csv(r'/kaggle/input/playground-series-s3e22/test.csv', index_col = 'id')
orig_train = pd.read_csv(r'/kaggle/input/horse-survival-dataset/horse.csv')

# Descriptive Statistics

Let's begin by taking a peek at our original training dataset first

In [ ]:
train.head(10)

In [ ]:
desc = pd.DataFrame(index = list(train))
desc['count'] = train.count()
desc['nunique'] = train.nunique()
desc['%unique'] = desc['nunique'] / len(train) * 100
desc['null'] = train.isnull().sum()
desc['type'] = train.dtypes
desc = pd.concat([desc, train.describe().T], axis = 1)
desc

So we have 1235 rows at most, with some of them having missing value... This won't be an easy competition for sure. Another interesting point is that most features are categorical. Even some numerical features can be considered as categorical.

In [ ]:
test.head(10)

In [ ]:
desc = pd.DataFrame(index = list(test))
desc['count'] = test.count()
desc['nunique'] = test.nunique()
desc['%unique'] = desc['nunique'] / len(test) * 100
desc['null'] = test.isnull().sum()
desc['type'] = test.dtypes
desc = pd.concat([desc, test.describe().T], axis = 1)
desc

Again, we have a lot of missing values in the test dataset. Another funny thing is that, we only have one unique value for `lesion_3` feature in the test dataset.

Let's try grouping the categorical and numerical features now.

In [ ]:
numerical_features = test._get_numeric_data().drop('lesion_3', axis = 1).columns
categorical_features = test.drop(numerical_features, axis = 1).drop('lesion_3', axis = 1).columns

# Adversarial Validation

The purpose of adversarial validation is to check whether train and test dataset have similar distribution or not. If the validation gives ROC-AUC score of close to .5, we can say that both datasets are similar. However, if it's far from .5, both dataset have different distribution.

The reason we want to do this is to make sure that we can trust our CV score, since a trusted CV only comes from dataset with similar distribution.

In [ ]:
def adversarial_validation(dataset_1 = train, dataset_2 = test, label = 'Train-Test'):

    adv_train = dataset_1.drop('outcome', axis = 1)#[dataset_1.lesion_3 == 0]
    adv_test = dataset_2.copy()

    adv_train['is_test'] = 0
    adv_test['is_test'] = 1

    adv = pd.concat([adv_train, adv_test], ignore_index = True)

    adv_shuffled = adv.sample(frac = 1)

    adv_X = adv_shuffled.drop('is_test', axis = 1)
    adv_y = adv_shuffled.is_test

    skf = StratifiedKFold(n_splits = 5, random_state = 42, shuffle = True)

    val_scores = []
    predictions = np.zeros(len(adv))

    for fold, (train_idx, val_idx) in enumerate(skf.split(adv_X, adv_y)):
    
        adv_lr = make_pipeline(OneHotEncoder(cols = categorical_features), XGBClassifier(random_state = 42))
        adv_lr.fit(adv_X.iloc[train_idx], adv_y.iloc[train_idx])
        
        val_preds = adv_lr.predict_proba(adv_X.iloc[val_idx])[:,1]
        predictions[val_idx] = val_preds
        val_score = roc_auc_score(adv_y.iloc[val_idx], val_preds)
        val_scores.append(val_score)
    
    fpr, tpr, _ = roc_curve(adv['is_test'], predictions)
    
    plt.figure(figsize = (10, 10), dpi = 300)
    sns.lineplot(x=[0, 1], y=[0, 1], linestyle="--", label="Indistinguishable Datasets")
    sns.lineplot(x=fpr, y=tpr, label="Adversarial Validation Classifier")
    plt.title(f'{label} Validation = {np.mean(val_scores):.5f}', weight = 'bold', size = 17)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.show()

In [ ]:
adversarial_validation()

The result is relatively close to .5, therefore we can trust our CV (albeit with caution due to the small dataset).

# Distribution of Numerical Features

Now that we have done taking a peek at the descriptive statistics and done adversarial validation of the datasets, let's try to see the feature distribution this time.

In [ ]:
fig, ax = plt.subplots(5, 2, figsize = (15, 20), dpi = 300)
ax = ax.flatten()

for i, column in enumerate(numerical_features):
    
    if i == 11:
        break
    
    sns.kdeplot(train[column], ax=ax[i], color=pal[0])
    sns.kdeplot(test[column], ax=ax[i], color=pal[2], warn_singular = False)
    sns.kdeplot(orig_train[column], ax=ax[i], color=pal[1])
    
    ax[i].set_title(f'{column} Distribution', size = 14)
    ax[i].set_xlabel(None)
    
fig.suptitle('Distribution of Feature\nper Dataset\n', fontsize = 24, fontweight = 'bold')
fig.legend(['Train', 'Test', 'Original Train'])
plt.tight_layout()

The train and test datasets have similar distribution as expected, with original dataset being somewhat different.

# Distribution of Categorical Features

Let's try checking the categorical features now.

In [ ]:
fig, ax = plt.subplots(16, 2, figsize = (16, 80), dpi = 300)
#ax = ax.flatten()

for i, column in enumerate(categorical_features):

    ax[i][0].pie(
        train[column].value_counts(), 
        shadow = True, 
        explode = [.1 for i in range(train[column].nunique())], 
        autopct = '%1.f%%',
        textprops = {'size' : 14, 'color' : 'white'}
    )

    sns.countplot(data = train, y = column, ax = ax[i][1], palette = 'viridis', order = train[column].value_counts().index)
    ax[i][1].yaxis.label.set_size(20)
    plt.yticks(fontsize = 12)
    ax[i][1].set_xlabel('Count in Train', fontsize = 15)
    ax[i][1].set_ylabel(f'{column}', fontsize = 15)
    plt.xticks(fontsize = 12)

fig.suptitle('Distribution of Categorical Features\nin Train Dataset\n\n\n\n', fontsize = 25, fontweight = 'bold')
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(16, 2, figsize = (16, 80), dpi = 300)
#ax = ax.flatten()

for i, column in enumerate(categorical_features):

    ax[i][0].pie(
        test[column].value_counts(), 
        shadow = True, 
        explode = [.1 for i in range(test[column].nunique())], 
        autopct = '%1.f%%',
        textprops = {'size' : 14, 'color' : 'white'}
    )

    sns.countplot(data = test, y = column, ax = ax[i][1], palette = 'viridis', order = test[column].value_counts().index)
    ax[i][1].yaxis.label.set_size(20)
    plt.yticks(fontsize = 12)
    ax[i][1].set_xlabel('Count in Test', fontsize = 15)
    ax[i][1].set_ylabel(f'{column}', fontsize = 15)
    plt.xticks(fontsize = 12)

fig.suptitle('Distribution of Categorical Features\nin Test Dataset\n\n\n\n', fontsize = 25, fontweight = 'bold')
plt.tight_layout()

There are... a lot of information to take in here. For example, we can see that most horses have taken surgeries before. Another one will be that most of the horses are adult.

Another peculiarity you might notice is that there are some categories in train dataset that aren't in test dataset. Features with such oddity are `peristalsis`, `nasogastric_reflux`, `rectal_exam_feces`. However, the missing values might be located within those null values across features. There is also the issue of one of the value in `pain` being different between training dataset (slight) and test dataset (moderate). We don't know if both values are supposed to exist or only one of them is.

# Target Distribution

We still need to check one last distribution: our target.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize = (16, 5))
ax = ax.flatten()

ax[0].pie(
    train['outcome'].value_counts(), 
    shadow = True, 
    explode = [.1 for i in range(train.outcome.nunique())], 
    autopct = '%1.f%%',
    textprops = {'size' : 14, 'color' : 'white'}
)

sns.countplot(data = train, y = 'outcome', ax = ax[1], palette = 'viridis', order = train['outcome'].value_counts().index)
ax[1].yaxis.label.set_size(20)
plt.yticks(fontsize = 12)
ax[1].set_xlabel('Count', fontsize = 20)
ax[1].set_ylabel(None)
plt.xticks(fontsize = 12)

fig.suptitle('Target Distribution', fontsize = 25, fontweight = 'bold')
plt.tight_layout()

We can see that most horses end up dead, either through euthanasia or through other causes. Only 46% of them are still living.

# Metrics

As a reminder, we are using F1-score with micro-averaging for metric. The formula for F1-score in general is as follows:

$$F_1 = 2 \frac{\mathrm{precision} \cdot \mathrm{recall}}{\mathrm{precision} + \mathrm{recall}} = \frac{2\mathrm{tp}}{2\mathrm{tp} + \mathrm{fp} + \mathrm{fn}}$$

Micro-averaging here means that we sum the true positive, false positive, and false negative from each class into one before calculating the F1-score. The code is as follows.

In [ ]:
def f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average = 'micro')

# Preprocessing

Because we know that there is only one unique value of `lesion_3` in test dataset, we can just remove any rows that have different value and drop the feature to make it simpler.

In [ ]:
preprocessed_train = train[train.lesion_3 == 0].drop('lesion_3', axis = 1)
preprocessed_test = test.drop('lesion_3', axis = 1)
preprocessed_orig_train = orig_train[orig_train.lesion_3 == 0].drop('lesion_3', axis = 1)

# Preparation

This is where we start preparing everything if we want to start building machine learning models.

In [ ]:
X = pd.concat([preprocessed_train, preprocessed_orig_train])
y = X.pop('outcome')

y_map = {k : v for v, k in enumerate(train.outcome.unique())}
y_reverse_map = {v : k for k, v in y_map.items()}

y = y.map(y_map)

f1_scorer = make_scorer(f1)

seed = 42
splits = 5
repeats = 4
rskf = RepeatedStratifiedKFold(n_splits = splits, random_state = seed, n_repeats = repeats)
np.random.seed(seed)

# Model

Let's start by evaluating the performance of our model first. To encode the categorical features, we will use One Hot Encoder. We will also use the original dataset in our training.

In [ ]:
def cross_val_score(estimator, cv = rskf, label = '', include_original = False):
    
    X = preprocessed_train.copy()
    y = X.pop('outcome')
    
    y = y.map(y_map)
    
    #initiate prediction arrays and score lists
    val_predictions = np.zeros((len(X)))
    #train_predictions = np.zeros((len(sample)))
    train_scores, val_scores = [], []
    
    #training model, predicting prognosis probability, and evaluating metrics
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        
        model = clone(estimator)
        
        #define train set
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]
        
        #define validation set
        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]
        
        if include_original:
            X_train = pd.concat([X_train, preprocessed_orig_train.drop('outcome', axis = 1)])
            y_train = pd.concat([y_train, preprocessed_orig_train.outcome.map(y_map)])
        
        #train model
        model.fit(X_train, y_train)
        
        #make predictions
        train_preds = model.predict(X_train)
        val_preds = model.predict(X_val)
                  
        #val_predictions[val_idx] += val_preds / repeats
        
        #evaluate model for a fold
        train_score = f1(y_train, train_preds)
        val_score = f1(y_val, val_preds)
        
        #append model score for a fold to list
        train_scores.append(train_score)
        val_scores.append(val_score)
    
    print(f'Val Score: {np.mean(val_scores):.5f} ± {np.std(val_scores):.5f} | Train Score: {np.mean(train_scores):.5f} ± {np.std(train_scores):.5f} | {label}')
    
    return val_scores, val_predictions

In [ ]:
score_list, oof_list = pd.DataFrame(), pd.DataFrame()

models = [
    ('log', LogisticRegression(random_state = seed, max_iter = 1000000)),
    ('svc', SVC(random_state = seed, probability = True)),
    ('lda', LinearDiscriminantAnalysis()),
    ('gnb', GaussianNB()),
    ('bnb', BernoulliNB()),
    ('knn', KNeighborsClassifier()),
    ('gauss', GaussianProcessClassifier(random_state = seed)),
    ('rf', RandomForestClassifier(random_state = seed)),
    ('et', ExtraTreesClassifier(random_state = seed)),
    ('xgb', XGBClassifier(random_state = seed)),
    ('lgb', LGBMClassifier(random_state = seed)),
    ('dart', LGBMClassifier(random_state = seed, boosting_type = 'dart')),
    ('cb', CatBoostClassifier(random_state = seed, verbose = 0)),
    ('gb', GradientBoostingClassifier(random_state = seed)),
    ('hgb', HistGradientBoostingClassifier(random_state = seed)),
]

for (label, model) in models:
    score_list[label], _ = cross_val_score(
        make_pipeline(OneHotEncoder(cols = categorical_features), SimpleImputer(), model),
        label = label,
        include_original = True
    )

In [ ]:
plt.figure(figsize = (8, 4), dpi = 300)
sns.barplot(data = score_list.reindex((-1 * score_list).mean().sort_values().index, axis = 1), palette = 'viridis', orient = 'h')
plt.title('Score Comparison', weight = 'bold', size = 20)
plt.show()

From the chart above, we can see that HistGradientBoostingRegressor gives the best result.

# Inference and Submission

Finally, let's train our chosen model on the whole train dataset and do inference on the test dataset. The chosen model will be HistGradientBoostingRegressor, with one-hot encoding and simple imputation inside the pipeline.

In [ ]:
model = make_pipeline(
    OneHotEncoder(cols = categorical_features),
    SimpleImputer(),
    HistGradientBoostingClassifier(random_state = seed)
)

model.fit(X, y)

In [ ]:
submission = preprocessed_test.copy()
submission['outcome'] = model.predict(submission)
submission.outcome = submission.outcome.map(y_reverse_map)

submission.outcome.to_csv('submission.csv')

Thanks for reading!